# Gemma 3 Feature 20947 — Phase 1

Single-feature dose response on `cuda:0`. The runner writes incremental JSON and Markdown results under `/kaggle/working`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

project_root = Path('/kaggle/working/phase1_sae_steering')
runner = project_root / 'experiments/single_feature/run_single_feature.py'
if not runner.is_file():
    matches = []
    for search_root in (Path('/kaggle/input'), Path.cwd()):
        if search_root.exists():
            matches.extend(
                p for p in search_root.rglob('run_single_feature.py')
                if p.parent.name == 'single_feature'
                and (p.parents[2] / 'features/feature_20947.json').is_file()
            )
        if matches:
            break
    if matches:
        project_root = matches[0].parents[2]
        runner = matches[0]
    else:
        subprocess.run(
            ['git', 'clone', '--depth', '1', 'https://github.com/dddddsz/phase1_sae_steering.git', str(project_root)],
            check=True,
        )
        if not runner.is_file():
            raise FileNotFoundError('Phase 1 project files were not found after git clone.')

print('Project root:', project_root)
print('Runner:', runner)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(project_root / 'requirements.txt')],
    check=True,
)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU is not available; enable a GPU accelerator.')
print('CUDA devices:', torch.cuda.device_count())
print('Using cuda:0:', torch.cuda.get_device_name(0))

if not os.environ.get('HF_TOKEN'):
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

ablation_runner = project_root / 'experiments/conditional_ablation/run_conditional_ablation.py'
output_dir = '/kaggle/working/phase1_sae_steering/outputs/conditional_ablation'
subprocess.run(
    [sys.executable, str(ablation_runner), '--exp', 'AB', '--output-dir', output_dir],
    check=True,
)